
# 🏥 AI Machine Health Monitor (Final Best Model)

**Overview:**
This notebook implements the final LSTM-based model for predicting machine health scores. It utilizes the best hyperparameters found during the optimization phase (Optuna Trial 2) to achieve high accuracy (R² > 0.90).


### 1. Imports & Device Configuration

In [ ]:

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
import pickle

# Set Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


### 2. Load Data

In [ ]:

# Load Dataset
# NOTE: Update the path below if necessary
file_path = r"C:\Users\Mostafa\Downloads\preprocessed_smart_data.csv"
df = pd.read_csv(file_path)

# Ensure correct data types and order
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['machine_id', 'timestamp']).reset_index(drop=True)

print(f"Data loaded: {len(df):,} samples | {df['machine_id'].nunique()} machines")


### 3. Construct Health Score (Target Variable)

In [ ]:

print("Building Health Score...")
sensor_cols = ['temperature', 'vibration', 'pressure', 'energy_consumption', 'humidity']

# Calculate Z-scores for each sensor per machine
for col in sensor_cols:
    df[f'{col}_z'] = df.groupby('machine_id')[col].transform(
        lambda x: np.abs((x - x.mean()) / (x.std() + 1e-8))
    )

# Sum Z-scores to get total degradation
df['degradation_score'] = df[[f'{c}_z' for c in sensor_cols]].sum(axis=1)

# Normalize to 0-100 (Inverted: High degradation = Low Health)
df['health_score'] = 100 - (
    (df['degradation_score'] - df['degradation_score'].min()) /
    (df['degradation_score'].max() - df['degradation_score'].min() + 1e-8)
) * 100

# Smooth the score using a rolling average
df['health_score'] = df.groupby('machine_id')['health_score'].transform(
    lambda x: x.rolling(20, min_periods=1).mean()
).clip(0, 100)

print("Health Score constructed successfully.")


### 4. Feature Engineering

In [ ]:

# Moving Averages (Trend)
df['temp_ma'] = df.groupby('machine_id')['temperature'].transform(lambda x: x.rolling(5, min_periods=1).mean())
df['vib_ma']  = df.groupby('machine_id')['vibration'].transform(lambda x: x.rolling(5, min_periods=1).mean())

# Rate of Change (Velocity of change)
df['temp_roc'] = df.groupby('machine_id')['temperature'].diff().fillna(0)
df['vib_roc']  = df.groupby('machine_id')['vibration'].diff().fillna(0)

# Drop initial NaNs generated by lag features
df = df.dropna().reset_index(drop=True)

# Define Feature list and Target
features = [
    'temperature', 'vibration', 'humidity', 'pressure', 'energy_consumption',
    'temp_ma', 'vib_ma', 'temp_roc', 'vib_roc'
]
target = 'health_score'


### 5. Model Architecture (Optimized)

In [ ]:

class FinalLSTMHealth(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=len(features),
            hidden_size=182,      # Optimized value
            num_layers=1,         # 1 layer performed better than 2
            batch_first=True,
            dropout=0.217         # Optimized dropout rate
        )
        self.fc = nn.Sequential(
            nn.Linear(182, 64),
            nn.ReLU(),
            nn.Dropout(0.217),
            nn.Linear(64, 1),
            nn.Sigmoid()          # Output between 0 and 1
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        # Scale output back to 0-100 range
        return self.fc(h_n[-1]) * 100


### 6. Data Preprocessing & Sequence Generation

In [ ]:

print("Preparing final dataset...")

# Scale Features
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

# Function to create sliding window sequences
def create_sequences(data_df, seq_len=20):
    X, y = [], []
    vals = data_df[features].values
    targets = data_df[target].values

    for i in range(seq_len, len(data_df)):
        X.append(vals[i-seq_len:i])
        y.append(targets[i])

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Generate Sequences
X_all, y_all = create_sequences(df_scaled, seq_len=20) # Optimized sequence length

print(f"Final sequences: {X_all.shape} | Labels: {y_all.shape}")


### 7. Training Configuration

In [ ]:

print("Initializing model...")

model = FinalLSTMHealth().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.00054) # Optimized Learning Rate
criterion = nn.L1Loss() # MAE Loss is robust for regression

# Create DataLoader
loader = DataLoader(
    TensorDataset(torch.tensor(X_all), torch.tensor(y_all).unsqueeze(1)),
    batch_size=64, # Optimized Batch Size
    shuffle=True
)


### 8. Training Loop

In [ ]:

print("Training started...")

for epoch in range(1, 81):
    model.train()
    total_loss = 0

    for x_b, y_b in loader:
        x_b, y_b = x_b.to(device), y_b.to(device)

        optimizer.zero_grad()
        pred = model(x_b)
        loss = criterion(pred, y_b)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch % 10 == 0) or (epoch == 80):
        print(f"Epoch {epoch:2d} | Avg Loss: {total_loss/len(loader):.4f}")


### 9. Evaluation

In [ ]:

# Split data (Last 20% for testing)
split = int(0.8 * len(X_all))
X_test, y_test = X_all[split:], y_all[split:]

model.eval()
with torch.no_grad():
    pred_test = model(torch.tensor(X_test).to(device))
    final_r2 = r2_score(y_test, pred_test.cpu().numpy())
    final_mae = mean_absolute_error(y_test, pred_test.cpu().numpy())

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE")
print("="*60)
print(f"   R² Score : {final_r2:.4f}")
print(f"   MAE      : {final_mae:.3f}")
print(f"   Model    : 1-layer LSTM (182 units)")
print(f"   Seq Len  : 20")
print(f"   Batch    : 64")
print(f"   LR       : 0.00054")
print("="*60)


### 10. Save Artifacts for Deployment

In [ ]:

# Save Model
torch.save(model.state_dict(), "FINAL_BEST_HEALTH_MODEL.pth")

# Save Scaler
with open("final_scaler_health.pkl", "wb") as f:
    pickle.dump(scaler, f)

# Optional: Save processed data for visualization
df.to_csv("data_with_health_score.csv", index=False)

print("\nMODEL SAVED SUCCESSFULLY!")
print("   → FINAL_BEST_HEALTH_MODEL.pth")
print("   → final_scaler_health.pkl")
print("   → data_with_health_score.csv")
print("\nReady for Streamlit deployment!")
